In [1]:
# Librerias
import numpy as np
from astropy.io import fits
import matplotlib.pyplot as plt
import astropy.units as u
from astropy.coordinates import SkyCoord
from astropy.wcs import WCS
import time

# import MUSE_utils.PPXF_astrometry_correction as ppxf_ac
# import MUSE_utils.PPXF_class as ppxf_c
# import MUSE_utils.PPXF_extinction_correction as ppxf_ec
import MUSE_utils.PPXF_fitting as ppxf_f
import MUSE_utils.PPXF_helper as ppxf_h
# import MUSE_utils.PPXF_redshift_calculator as ppxf_rc
# import MUSE_utils.PPXF_zapping as ppxf_z

import joblib
from tqdm import tqdm
from joblib import Parallel, delayed

d:\Universidad\Magister\MUSE PKS0745\MUSE_utils\PPXF_fitting.py:261: SyntaxWarning: invalid escape sequence '\s'
  txt = f"Voronoi bin {j+1} / {nbins}; SPS: {sps_name}; $\sigma$={sigbin[j]:.0f} km/s; S/N={pp_stars.sn:.1f}"
d:\Universidad\Magister\MUSE PKS0745\MUSE_utils\PPXF_fitting.py:20: UserWarning: ⚠️ VorBin is deprecated and superseded by PowerBin https://pypi.org/project/powerbin/ 
  from vorbin.voronoi_2d_binning import voronoi_2d_binning


In [ ]:
t0 = time.time()
gal_NED_id = "MUSE0745"
name_cube = "PKS0745_cube_wcscorr_ext-cor_small.fits"
with fits.open(name_cube) as hdu:
    header_data= hdu[1].header
vvcs= WCS(header_data)
coord = vvcs.pixel_to_world(header_data["NAXIS1"],header_data["NAXIS2"],0)

z = 0.10246 
c = SkyCoord(ra=116.87909416*u.degree, dec=-19.29329137*u.degree, frame="icrs")

lam_range_temp = [4600, 7000]

s = ppxf_h.read_data_cube(name_cube, lam_range_temp, z)#, [75,330],[100,349])

s.outfolder = "./prueba/"
s.start = [0, 200.]    # (km/s), starting guess for [V,sigma]
print(time.time()-t0)

In [ ]:
t0 = time.time()
gal_NED_id = "Abell"
name_cube = "Abell1795_corr.fits"
with fits.open(name_cube) as hdu:
    header_data= hdu[1].header

z =  0.06267
lam_range_temp = [4600, 7000]

s = ppxf_h.read_data_cube(name_cube, lam_range_temp, z, [75,330],[99,349])#[75,330],[100,349]# 84->75 #130,200 250,300

s.outfolder = "./maps_ABELL/"
s.start = [0, 200.]    # (km/s), starting guess for [V,sigma]
print(time.time()-t0)

In [ ]:
t0 = time.time()
valid = np.isfinite(s.noise)
median_noise = np.nanmedian(s.noise[valid])
s.noise[~valid] = median_noise

valid = np.isfinite(s.signal)
median_s = np.nanmedian(s.signal[valid])
s.signal[~valid] = median_s
print(time.time()-t0)

In [ ]:
t0 = time.time()
s.redshift = z

### 1. Voronoi binning and stellar kinematics
sk = ppxf_f.stellar_kinematics(
    s=s,
    target_sn=10,
    noise=s.noise,
    signal=s.signal,
    width_lines=1200,  # width to use to mask the emission lines
)

stars = sk.stellar_kinematics()
print(time.time()-t0)

In [ ]:
t0 = time.time()
fit_gas = ppxf_f.GasKinematicsFitter(
    s=s,
    sps=stars.sps,
    lam_gal=stars.lam_gal,
    velbin=stars.velbin,
    sigbin=stars.sigbin,
    h3=stars.h3,
    h4=stars.h4,
    bin_num=stars.bin_num,
    optimal_templates=stars.optimal_templates
)

fit_gas.Fit_cube_one_component(n_jobs=8)
fit_gas.Save_all_emission_lines_one_component()
print(time.time()-t0)

In [ ]:
#joblib.dump(fit_gas, 'fit_gas_obj.joblib', compress=3)

In [2]:
fit_gas = joblib.load('fit_gas_obj.joblib') #2'26"

In [ ]:
t0 = time.time()
# Run all spaxels two gaussians
fit_gas.Fit_cube(n_jobs=10, batch_size=22, max_nbytes="5M")
# Build flux maps, component maps, weighted velocities
fit_gas.Process_all_lines()
print(time.time()-t0)

In [3]:
t0 = time.time()
#Create clipped maps for each gas component of flux, velocity and sigma
fit_gas.Clip_maps(name_fits_properties = "maps_ABELL\MUSE_PKS0745_properties.fits")

print(time.time()-t0)


Map_clipping line: Hbeta

Map_clipping line: Halpha

Map_clipping line: [SII]6716

Map_clipping line: [SII]6731

Map_clipping line: [NII]6548

Map_clipping line: [NII]6584

Map_clipping line: [OIII]4959

Map_clipping line: [OIII]5007

Map_clipping line: HeII4687

Map_clipping line: HeI5876

Map_clipping line: [NII]5755

Map_clipping line: [NI]5198

Map_clipping line: [NI]5200

Map_clipping line: [OI]6300_d
56.39512085914612
